In [ ]:
!pip uninstall torchao -y

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip install peft -q
from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
import pandas as pd
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, AutoModelForSequenceClassification, AutoTokenizer, set_seed, TrainerCallback
from datasets import Dataset
import joblib
import numpy as np
from google.colab import drive
import os
import json
import zipfile
from sklearn.metrics import f1_score, classification_report
import time
import torch
from transformers.trainer_utils import get_last_checkpoint

In [ ]:
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/thesis_results/AAPD_ModernBERT_LoRa"
os.makedirs(output_dir, exist_ok=True)

Mounted at /content/drive


**Dataset loading**

In [ ]:
with zipfile.ZipFile("aapd.zip") as z:
    with z.open("aapd.json") as f:
        aapd = json.load(f)

In [ ]:
aapd_df_train = pd.DataFrame(aapd["data"]["train"])
aapd_df_val = pd.DataFrame(aapd["data"]["val"])
aapd_df_test = pd.DataFrame(aapd["data"]["test"])

In [ ]:
mlb = joblib.load("mlb.joblib")

In [ ]:
#reusing the aapd's mlb
aapd_y_train = mlb.transform(aapd_df_train["labels"])
aapd_y_val   = mlb.transform(aapd_df_val["labels"])
aapd_y_test  = mlb.transform(aapd_df_test["labels"])

In [ ]:
aapd_y_train.shape, aapd_y_val.shape, aapd_y_test.shape #ok

((53840, 54), (1000, 54), (1000, 54))

In [ ]:
aapd_X_train = aapd_df_train["text"]
aapd_X_val   = aapd_df_val["text"]
aapd_X_test  = aapd_df_test["text"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [ ]:
def tokenize(texts):
    return tokenizer(texts.tolist(), padding="max_length", truncation=True, max_length=512)

train_enc = tokenize(aapd_X_train)
dev_enc   = tokenize(aapd_X_val)
test_enc  = tokenize(aapd_X_test)

In [ ]:
y_train_bin = aapd_y_train.astype(np.float32)
y_dev_bin   = aapd_y_val.astype(np.float32)
y_test_bin  = aapd_y_test.astype(np.float32)

print(mlb.classes_)
print(y_train_bin.shape)

['Adaptation and Self-Organizing Systems' 'Applications'
 'Artificial Intelligence' 'Combinatorics' 'Computation and Language'
 'Computational Complexity'
 'Computational Engineering, Finance, and Science'
 'Computational Geometry' 'Computational Linguistics'
 'Computer Science and Game Theory'
 'Computer Vision and Pattern Recognition' 'Computers and Society'
 'Cryptography and Security' 'Data Analysis, Statistics and Probability'
 'Data Structures and Algorithms' 'Databases' 'Digital Libraries'
 'Discrete Mathematics' 'Disordered Systems and Neural Networks'
 'Distributed, Parallel, and Cluster Computing'
 'Formal Languages and Automata Theory' 'Human-Computer Interaction'
 'Information Retrieval' 'Information Theory (Computer Science)'
 'Information Theory (Mathematics)' 'Logic' 'Logic in Computer Science'
 'Machine Learning (Computer Science)' 'Machine Learning (Statistics)'
 'Mathematical Software' 'Methodology' 'Multiagent Systems' 'Multimedia'
 'Networking and Internet Architect

In [ ]:
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}

In [ ]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "labels": y_train_bin.astype("float32")})

eval_dataset = Dataset.from_dict({
    "input_ids": dev_enc["input_ids"],
    "attention_mask": dev_enc["attention_mask"],
    "labels": y_dev_bin.astype("float32")})

test_dataset = Dataset.from_dict({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "labels": y_test_bin.astype("float32")})

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   #sigmoid
    preds = (probs >= 0.5).astype(int) #default threshold

    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro}

In [ ]:
print("Train labels:", y_train_bin.shape)
print("Val labels:", y_dev_bin.shape)
print("Test labels:", y_test_bin.shape)
print("Number of labels:", len(mlb.classes_))

Train labels: (53840, 54)
Val labels: (1000, 54)
Test labels: (1000, 54)
Number of labels: 54


**Training function**

In [ ]:
#helpers for counting times for the final run with ModernBERT, as the run would get disconnected by collab  since it takes a long time to run it with 10 epochs
class CheckpointTimeCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.time_log_path = os.path.join(output_dir, "time_log.json")
        self.previous_time_sec = 0.0
        self.session_start = None
        self.current_total_time_sec = 0.0
        self.current_session_time_sec = 0.0

    def on_train_begin(self, args, state, control, **kwargs):
        if os.path.exists(self.time_log_path):
            with open(self.time_log_path, "r") as f:
                self.previous_time_sec = json.load(f).get("train_time_sec", 0.0)
        else:
            self.previous_time_sec = 0.0

        self.session_start = time.perf_counter()
        self.current_total_time_sec = self.previous_time_sec
        self.current_session_time_sec = 0.0

    def _save_time(self, state):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        self.current_session_time_sec = time.perf_counter() - self.session_start
        self.current_total_time_sec = self.previous_time_sec + self.current_session_time_sec

        data = {
            "train_time_sec": self.current_total_time_sec,
            "current_session_train_time_sec": self.current_session_time_sec,
            "previous_train_time_sec": self.previous_time_sec,
            "last_global_step": int(state.global_step),
            "last_epoch": float(state.epoch) if state.epoch is not None else None,
        }

        os.makedirs(self.output_dir, exist_ok=True)

        tmp_path = self.time_log_path + ".tmp"
        with open(tmp_path, "w") as f:
            json.dump(data, f, indent=2)

        os.replace(tmp_path, self.time_log_path)

    def on_save(self, args, state, control, **kwargs):
        self._save_time(state)

    def on_train_end(self, args, state, control, **kwargs):
        self._save_time(state)

    def get_times(self):
        if os.path.exists(self.time_log_path):
            with open(self.time_log_path, "r") as f:
                data = json.load(f)

            return (
                data.get("train_time_sec", self.current_total_time_sec),
                data.get("current_session_train_time_sec", self.current_session_time_sec),
            )

        return self.current_total_time_sec, self.current_session_time_sec

In [ ]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

#measure vram only for final best config
def get_peak_vram_gb():
    if not torch.cuda.is_available():
        return None

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(config, seed=0, evaluate_test=False, measure_vram=False, save_report=False):
    set_seed(seed)

    if measure_vram:
        reset_cuda_peak_memory()

    model = AutoModelForSequenceClassification.from_pretrained(
        config["base_model"],
        num_labels=len(mlb.classes_),
        problem_type="multi_label_classification",
        id2label=id2label,
        label2id=label2id)

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        target_modules=config["target_modules"],
        modules_to_save= config["modules_to_save"])
    model = get_peft_model(model, peft_config)


    #for the final statistics
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=config["output_dir"],

        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],

        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        save_total_limit=2,
        fp16=True,
        report_to="none",
        label_names=["labels"])


    time_callback = CheckpointTimeCallback(config["output_dir"])

    #standard trainer for LoRA
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"]), time_callback])


    #for resuming if something goes wrong//collab's runtime gets disconnected
    last_checkpoint = None
    if os.path.isdir(config["output_dir"]):
        last_checkpoint = get_last_checkpoint(config["output_dir"])

    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("Starting training from scratch.")


####imporved for the final run on MODERNBERT
    sync_cuda()
    # includes training + epoch validation + checkpoint saving + early stopping + loading best model
    trainer.train(resume_from_checkpoint=last_checkpoint)
    sync_cuda()
    # accumulated training time saved after completed checkpoints/epochs
    train_time_sec, current_session_train_time_sec = time_callback.get_times()


    if measure_vram:
        training_peak_vram_gb = get_peak_vram_gb()
    else:
        training_peak_vram_gb = None

    sync_cuda()
    val_start = time.perf_counter()

    val_results = trainer.evaluate(eval_dataset, metric_key_prefix="val")

    sync_cuda()
    val_eval_time_sec = time.perf_counter() - val_start

    result = {
        "model": config["base_model"],
        "dataset": "AAPD",
        "method": "lora",
        "seed": seed,

        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "num_train_epochs": config["num_train_epochs"],

        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "actual_epochs_trained": trainer.state.epoch,


        "train_time_sec": train_time_sec,
        "current_session_train_time_sec": current_session_train_time_sec,
        "val_eval_time_sec": val_eval_time_sec,

        "training_peak_vram_gb": training_peak_vram_gb,

        "trainable_params": trainable_params,
        "total_params": total_params,

        "val_f1_macro": val_results["val_f1_macro"],
        "val_f1_micro": val_results["val_f1_micro"]}

    if evaluate_test:
        sync_cuda()
        test_start = time.perf_counter()

        #single forward pass, gives metrics + raw predictions
        test_pred_output = trainer.predict(test_dataset)

        sync_cuda()
        test_eval_time_sec = time.perf_counter() - test_start

        #derive predictions, needed for classification report
        test_probs = 1 / (1 + np.exp(-test_pred_output.predictions))
        test_binary_preds = (test_probs >= 0.5).astype(int)
        gold_labels = (test_pred_output.label_ids >= 0.5).astype(int)

        test_metrics = test_pred_output.metrics

        result.update({
            "test_eval_time_sec": test_eval_time_sec,
            "test_inference_per_sample_ms": (test_eval_time_sec / len(test_dataset)) * 1000,
            "test_f1_macro": test_metrics["test_f1_macro"],
            "test_f1_micro": test_metrics["test_f1_micro"],
            "avg_predicted_labels": float(test_binary_preds.sum(axis=1).mean()),
            "avg_gold_labels": float(gold_labels.sum(axis=1).mean()),})

        if save_report:
            report_dict = classification_report(
                gold_labels, test_binary_preds,
                target_names=mlb.classes_, zero_division=0, output_dict=True)
            report_df = pd.DataFrame(report_dict).T
            report_path = os.path.join(output_dir, f"classification_report_seed_{seed}.csv")
            report_df.to_csv(report_path)
            print(f"Classification report saved to {report_path}")

            #saving raw arrays for possible future analysis
            predictions_path = os.path.join(
                output_dir,
                f"test_predictions_seed_{seed}.npz")
            np.savez_compressed(
                predictions_path,
                y_true=gold_labels,
                y_pred=test_binary_preds,
                y_prob=test_probs,
                label_names=np.array(mlb.classes_),
                threshold=np.array([0.5]))
            result["test_predictions_path"] = predictions_path
            print(f"Predictions saved to {predictions_path}")

    result["total_measured_time_sec"] = (result["train_time_sec"] + result["val_eval_time_sec"] + result.get("test_eval_time_sec", 0))

    return result

In [ ]:
#fixed params
base_config = {
    "output_dir": os.path.join(output_dir, "search"),
    "base_model": "answerdotai/ModernBERT-base",
    "tokenizer_name": "answerdotai/ModernBERT-base",

    "max_length": 512,
    "num_train_epochs": 4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "early_stopping_patience": 3,

    #lora params
    "lora_r": 8,   #in Razuvayevskaya et al. (2024) they showed that this param value doesnt matter
    "lora_alpha": 8,          # alpha = r as in Razuvayevskaya et al. (2024)
    "lora_dropout": 0.05,     # as in Razuvayevskaya et al. (2024)
    "target_modules": ["Wqkv", "Wo", "Wi"], #Razuvayevskaya et al. (2024) - all layers
    "modules_to_save": ["head", "classifier"]}

#small search on the most relevant hyperparameters
learning_rates = [1e-4, 3e-4, 5e-4]     #FFT:[1e-5, 2e-5, 3e-5]
batch_sizes = [8, 16]


**Hyperparameter search**

In [ ]:

search_results_path = os.path.join(output_dir, "search_results.csv")
search_results = []

for lr in learning_rates:
    for bs in batch_sizes:
        config = base_config.copy()
        config["learning_rate"] = lr
        config["batch_size"] = bs
        config["output_dir"] = (f"{base_config['output_dir']}/lr_{lr}_bs_{bs}")

        #skipingp already-completed configs on resume
        if os.path.exists(search_results_path):
            existing = pd.read_csv(search_results_path)
            already_done = existing[
                (existing["learning_rate"] == lr) &
                (existing["batch_size"] == bs)]
            if len(already_done) > 0:
                print(f"Skipping lr={lr}, bs={bs} (already done)")
                search_results.append(already_done.iloc[0].to_dict())
                continue

        print("=" * 80)
        print(f"Running ModernBERT: lr={lr}, batch_size={bs}")
        print("=" * 80)

        result = run_training(config, seed=0)
        search_results.append(result)

        #saving incrementally after every config
        pd.DataFrame(search_results).to_csv(search_results_path, index=False)

search_results_df = pd.DataFrame(search_results)
search_results_df = search_results_df.sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
search_results_df.to_csv(search_results_path, index=False)

#!!! train_time_sec in search results is unreliable due to checkpoint resumption !!!
#!!! timing is only reported from the final seed runs - I ensured the run is not resumed

#the warining about early stopping can also be ignored, it works

search_results_df

Skipping lr=0.0001, bs=8 (already done)
Skipping lr=0.0001, bs=16 (already done)
Skipping lr=0.0003, bs=8 (already done)
Skipping lr=0.0003, bs=16 (already done)
Skipping lr=0.0005, bs=8 (already done)
Skipping lr=0.0005, bs=16 (already done)


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,train_time_sec,val_eval_time_sec,training_peak_vram_gb,trainable_params,total_params,val_f1_macro,val_f1_micro,total_measured_time_sec
0,answerdotai/ModernBERT-base,AAPD,lora,0,0.0005,8,4,/content/drive/MyDrive/thesis_results/AAPD_Mod...,0.593458,4.0,11697.166749,21.960993,NaN,2321718,151968108,0.593458,0.753073,11719.127742
1,answerdotai/ModernBERT-base,AAPD,lora,0,0.0005,16,4,/content/drive/MyDrive/thesis_results/AAPD_Mod...,0.585486,4.0,11444.327855,21.975792,NaN,2321718,151968108,0.585486,0.752413,11466.303647
2,answerdotai/ModernBERT-base,AAPD,lora,0,0.0003,8,4,/content/drive/MyDrive/thesis_results/AAPD_Mod...,0.577110,4.0,11866.345167,22.274906,NaN,2321718,151968108,0.577110,0.755116,11888.620073
3,answerdotai/ModernBERT-base,AAPD,lora,0,0.0003,16,4,/content/drive/MyDrive/thesis_results/AAPD_Mod...,0.561710,4.0,11120.878667,21.043500,NaN,2321718,151968108,0.561710,0.747553,11141.922167
4,answerdotai/ModernBERT-base,AAPD,lora,0,0.0001,8,4,/content/drive/MyDrive/thesis_results/AAPD_Mod...,0.540449,4.0,2979.143691,22.225250,NaN,2321718,151968108,0.540449,0.739241,3001.368941
5,answerdotai/ModernBERT-base,AAPD,lora,0,0.0001,16,4,/content/drive/MyDrive/thesis_results/AAPD_Mod...,0.517375,4.0,11267.460104,21.618173,NaN,2321718,151968108,0.517375,0.724178,11289.078277


**Best configuration**

In [ ]:
best_row = search_results_df.iloc[0]

best_lr = float(best_row["learning_rate"])
best_batch_size = int(best_row["batch_size"])

print("Best learning rate:", best_lr)
print("Best batch size:", best_batch_size)
print("Best validation macro-F1:", best_row["val_f1_macro"])
print("Best checkpoint:", best_row["best_checkpoint"])

Best learning rate: 0.0005
Best batch size: 8
Best validation macro-F1: 0.5934581222168666
Best checkpoint: /content/drive/MyDrive/thesis_results/AAPD_ModernBERT_LoRa/search/lr_0.0005_bs_8/checkpoint-26920


In [ ]:
best_config = base_config.copy()
best_config["learning_rate"] = best_lr
best_config["batch_size"] = best_batch_size
best_config["selection_metric"] = "val_f1_macro"
best_config["best_validation_macro_f1"] = float(best_row["val_f1_macro"])
best_config["best_validation_micro_f1"] = float(best_row["val_f1_micro"])
best_config["best_checkpoint_from_search"] = best_row["best_checkpoint"]

best_config_path = os.path.join(output_dir, "best_config.json")

with open(best_config_path, "w") as f:
    json.dump(best_config, f, indent=2)


**Final run (test set) on the best found configuration**

In [ ]:
with open(best_config_path, "r") as f:
    final_config = json.load(f)

#final runs use the full training budget with early stopping
final_config["num_train_epochs"] = 10

In [ ]:
test_output_dir = os.path.join(output_dir, "test")
os.makedirs(test_output_dir, exist_ok=True)

test_results_path = os.path.join(test_output_dir, "AAPD_ModernBERT_LoRa_test_results.csv")
test_results = []

for seed in [0, 1, 2]:
    config = final_config.copy()
    config["seed"] = seed
    config["output_dir"] = (os.path.join(test_output_dir, f"AAPD_ModernBERT_LoRa_test_seed_{seed}"))

    #skip already-completed seeds on resume
    if os.path.exists(test_results_path):
        existing = pd.read_csv(test_results_path)
        already_done = existing[existing["seed"] == seed]
        if len(already_done) > 0:
            print(f"Skipping seed={seed} (already done)")
            test_results.append(already_done.iloc[0].to_dict())
            continue

    print("=" * 80)
    print(f"Final run: seed={seed}, lr={final_config['learning_rate']}, batch_size={final_config['batch_size']}")
    print("=" * 80)
    result = run_training(config, seed=seed, evaluate_test=True, measure_vram=True, save_report = True)
    test_results.append(result)

    #save incrementally after every seed
    pd.DataFrame(test_results).to_csv(test_results_path, index=False)

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(test_results_path, index=False)
test_results_df

Skipping seed=0 (already done)
Skipping seed=1 (already done)
Final run: seed=2, lr=0.0005, batch_size=8


model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training from scratch.


...

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.100206,0.069689,0.677004,0.430350
2,0.065445,0.062043,0.715829,0.502699
3,0.059386,0.058670,0.752879,0.560715
4,0.053205,0.057426,0.751812,0.575334
5,0.046709,0.058782,0.746869,0.579401
6,0.038788,0.062030,0.748523,0.576546
7,0.030251,0.070123,0.742314,0.579201
8,0.021993,0.082916,0.741753,0.571853


[transformers] early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro
0.021993,0.058782,8,0.746869,0.579401


Classification report saved to /content/drive/MyDrive/thesis_results/AAPD_ModernBERT_LoRa/classification_report_seed_2.csv
Predictions saved to /content/drive/MyDrive/thesis_results/AAPD_ModernBERT_LoRa/test_predictions_seed_2.npz


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,...,val_f1_macro,val_f1_micro,test_eval_time_sec,test_inference_per_sample_ms,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,test_predictions_path,total_measured_time_sec
0,answerdotai/ModernBERT-base,AAPD,lora,0,0.0005,8,10,/content/drive/MyDrive/thesis_results/AAPD_Mod...,0.585770,10.0,...,0.585770,0.746069,22.879535,22.879535,0.572529,0.721911,2.225,2.421,/content/drive/MyDrive/thesis_results/AAPD_Mod...,30223.249817
1,answerdotai/ModernBERT-base,AAPD,lora,1,0.0005,8,10,/content/drive/MyDrive/thesis_results/AAPD_Mod...,0.590643,8.0,...,0.590643,0.747569,22.921884,22.921884,0.590067,0.733173,2.140,2.421,/content/drive/MyDrive/thesis_results/AAPD_Mod...,24192.527161
2,answerdotai/ModernBERT-base,AAPD,lora,2,0.0005,8,10,/content/drive/MyDrive/thesis_results/AAPD_Mod...,0.579401,8.0,...,0.579401,0.746869,22.543871,22.543871,0.580345,0.737980,2.113,2.421,/content/drive/MyDrive/thesis_results/AAPD_Mod...,24100.896865


In [ ]:
test_summary_df = test_results_df[[
    "test_f1_macro",
    "test_f1_micro",
    "avg_predicted_labels",
    "avg_gold_labels",
    "train_time_sec",
    "val_eval_time_sec",
    "test_eval_time_sec",
    "test_inference_per_sample_ms",
    "training_peak_vram_gb",
    "actual_epochs_trained",
    "trainable_params", #doesnt need mean across seeds, but this doesnt matter, should be the same
    "total_params",
    "total_measured_time_sec"]].agg(["mean", "std"])

test_summary_path =  os.path.join(test_output_dir, "AAPD_ModernBERT_LoRa_test_results_summary.csv")
test_summary_df.to_csv(test_summary_path)

test_summary_df

,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,train_time_sec,val_eval_time_sec,test_eval_time_sec,test_inference_per_sample_ms,training_peak_vram_gb,actual_epochs_trained,trainable_params,total_params,total_measured_time_sec
mean,0.580980,0.731021,2.159333,2.421,26126.586468,22.856383,22.781763,22.781763,3.750548,8.666667,2321718.0,151968108.0,26172.224614
std,0.008786,0.008247,0.058449,0.000,3508.436006,0.160553,0.207106,0.207106,0.005522,1.154701,0.0,0.0,3508.589877
